In [ ]:
from utils import get_dataset_lines
from math import dist

# Farthest First Traversal

Although the $k$-Center Clustering Problem is easy to state, it is NP-Hard. The Farthest First Traversal heuristic, whose pseudocode is shown below, selects centers from the points in $Data$ (instead of from all possible points in $m$-dimensional space). It begins by selecting an arbitrary point in $Data$ as the first center and iteratively adds a new center as the point in $Data$ that is farthest from the centers chosen so far, with ties broken arbitrarily (see figure below).

    FarthestFirstTraversal(Data, k)
        Centers <- the set consisting of a single randomly chosen point from Data
        while |Centers| < k
            DataPoint <- the point in Data maximizing d(DataPoint, Centers)
            add DataPoint to Centers
        return Centers

**Code Challenge**: Implement the FarthestFirstTraversal clustering heuristic.

**Input**: Integers $k$ and $m$ followed by a set of points $Data$ in $m$-dimensional space.

**Output**: A set $Centers$ consisting of $k$ points (centers) resulting from applying FarthestFirstTraversal($Data$, $k$), where the first point from $Data$ is chosen as the first center to initialize the algorithm.

**Sample Input**:

```
3 2
0.0 0.0
5.0 5.0
0.0 5.0
1.0 1.0
2.0 2.0
3.0 3.0
1.0 2.0
```

**Sample Output**:

```
0.0 0.0
5.0 5.0
0.0 5.0
```

In [61]:
def DistanceToCenter(point, centers):
    min = float('inf')
    for center in centers:
        distance = dist(point, center)
        if distance < min:
            min = distance
    return min

def FurthestPointFromCenter(data, centers):
    furthestDistance = 0
    for point in data:
        distance = DistanceToCenter(point, centers)
        if distance > furthestDistance:
            furthestPoint = point
            furthestDistance = distance
    return furthestPoint

def FarthestFirstTraversal(data, k):
    centers = [data[0]]
    while len(centers) < k:
        dataPoint = FurthestPointFromCenter(data, centers)
        centers.append(dataPoint)
    return centers                

In [62]:
# Sample Input
k = 3
data = [
    (0.0, 0.0),
    (5.0, 5.0),
    (0.0, 5.0),
    (1.0, 1.0),
    (2.0, 2.0),
    (3.0, 3.0),
    (1.0, 2.0)
]

print(FarthestFirstTraversal(data,k))

[(0.0, 0.0), (5.0, 5.0), (0.0, 5.0)]


In [64]:
# Test Dataset
test_dataset_filename = 'dataset_30181_2.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    k = int(lines[0].split()[0])
    data = []
    for line in lines[1:]:
        point = list(map(float, line.split()))
        data.append(point)

    centers = FarthestFirstTraversal(data, k)
    for center in centers:
        print(" ".join(map(str, center)))
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")


0.9 4.1 5.7 14.9 1.7
13.5 2.9 38.9 4.9 17.6
2.6 34.4 3.8 12.4 25.0
35.4 10.3 13.6 3.4 1.4
11.0 18.0 6.2 45.6 8.5
17.6 6.2 6.2 6.1 32.0
6.2 31.0 22.9 7.8 2.9


# Squared Error Distortion

To address limitations of MaxDistance, we will introduce a new scoring function. Given a set $Data$ of $n$ data points and a set $Centers$ of $k$ centers, the squared error distortion of $Data$ and $Centers$, denoted $Distortion(Data, Centers)$, is defined as the mean squared distance from each data point to its nearest center,

$$Distortion(Data,Centers) = (1/n) \sum_{\text{all points } DataPoint \text{ in } Data} d(DataPoint, Centers)^2 .$$

Note that whereas $MaxDistance(Data, Centers)$ only accounts for the length of the single longest segment, the squared error distortion accounts for the length of all segments.

**Code Challenge**: Solve the Squared Error Distortion Problem.

**Input**: Integers $k$ and $m$, followed by a set of centers $Centers$ and a set of points $Data$.

**Output**: The squared error distortion $Distortion(Data, Centers)$.

**Sample Input**:

```
2 2
2.31 4.55
5.96 9.08
--------
3.42 6.03
6.23 8.25
4.76 1.64
4.47 4.33
3.95 7.61
8.93 2.97
9.74 4.03
1.73 1.28
9.72 5.01
7.27 3.77
```

**Sample Output**:

```
18.246
```

In [65]:
def Distortion(centers, data):
    return sum(map(lambda x: DistanceToCenter(x, centers) ** 2, data))/len(data)

In [67]:
# Sample Input
centers = [
    (2.31, 4.55),
    (5.96, 9.08)
]
data = [
    (3.42, 6.03),
    (6.23, 8.25),
    (4.76, 1.64),
    (4.47, 4.33),
    (3.95, 7.61),
    (8.93, 2.97),
    (9.74, 4.03),
    (1.73, 1.28),
    (9.72, 5.01),
    (7.27, 3.77)
]

print(Distortion(centers, data))

18.245559999999998


In [70]:
# Test Dataset
test_dataset_filename = 'dataset_30170_3.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    k, m = map(int, lines[0].split())
    sep_index = next(i for i, line in enumerate(lines) if set(line) == {'-'})
    centers = [tuple(map(float, line.split())) for line in lines[1:sep_index]]
    data = [tuple(map(float, line.split())) for line in lines[sep_index + 1:]]
    
    print(Distortion(centers, data))
    
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")


33.904153046062405


# Lloyd Algorithm for k-Means Clustering

**Code Challenge**: Implement the Lloyd algorithm for $k$-means clustering.

**Input**: Integers $k$ and $m$ followed by a set of points $Data$ in $m$-dimensional space.

**Output**: A set $Centers$ consisting of $k$ points ($centers$) resulting from applying the Lloyd algorithm to $Data$ and $Centers$, where the first $k$ points from $Data$ are selected as the first $k$ centers. You should report your answers to at least three decimal points.

**Sample Input**:

```
2 2
1.3 1.1
1.3 0.2
0.6 2.8
3.0 3.2
1.2 0.7
1.4 1.6
1.2 1.0
1.2 1.1
0.6 1.5
1.8 2.6
1.2 1.3
1.2 1.0
0.0 1.9
```

**Sample Output**:

```
1.800 2.867
1.060 1.140
```

In [71]:
def AssignPointsToClusters(data, centers):
    clusters = [[] for _ in range(len(centers))]
    for point in data:
        min_dist = float('inf')
        closest_center_index = -1
        for i, center in enumerate(centers):
            d = dist(point, center)
            if d < min_dist:
                min_dist = d
                closest_center_index = i
        clusters[closest_center_index].append(point)
    return clusters

def LloydsAlgorithm(data, k, m):
    # Select first k points as initial centers
    centers = data[:k]
    
    while True:
        clusters = AssignPointsToClusters(data, centers)
        
        new_centers = []
        for i, cluster in enumerate(clusters):
            if not cluster:
                new_centers.append(centers[i]) # Keep old center if cluster is empty
            else:
                n = len(cluster)
                centroid = tuple(sum(p[dim] for p in cluster) / n for dim in range(m))
                new_centers.append(centroid)
        
        if centers == new_centers:
            break
            
        centers = new_centers
        
    return centers

In [72]:
# Sample Input
k = 2
m = 2
sample_data = [
    (1.3, 1.1), (1.3, 0.2), (0.6, 2.8), (3.0, 3.2),
    (1.2, 0.7), (1.4, 1.6), (1.2, 1.0), (1.2, 1.1),
    (0.6, 1.5), (1.8, 2.6), (1.2, 1.3), (1.2, 1.0),
    (0.0, 1.9)
]

final_centers = LloydsAlgorithm(sample_data, k, m)
for center in final_centers:
    print(f"{center[0]:.3f} {center[1]:.3f}")

# Expected Output:
# 1.800 2.867
# 1.060 1.140

1.800 2.867
1.060 1.140


In [73]:
# Test Dataset
test_dataset_filename = 'dataset_30171_3.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    if lines:
        k, m = map(int, lines[0].split())
        data = []
        for line in lines[1:]:
            if line.strip(): # Check for non-empty lines
                point = tuple(map(float, line.split()))
                data.append(point)

        final_centers = LloydsAlgorithm(data, k, m)
        for center in final_centers:
            print(" ".join(f"{x:.3f}" for x in center))
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

5.216 4.554 3.439
9.906 8.471 19.679
16.437 7.387 5.938
6.384 16.633 5.881
4.324 5.105 11.922


In [77]:
# Coursera Quiz Questions

# Q6 Compute MaxDistance(Data, Centers) for the following Data and Centers:

Data = [(2, 8), (2, 5), (6, 9), (7, 5), (5, 2)] 

Centers = [(3, 5), (5, 4)]

furthest = FurthestPointFromCenter(Data, Centers)
print("Q6:", DistanceToCenter(furthest, Centers))

# Q7 Compute Distortion(Data, Centers) for the following Data and Centers:

Data = [(2, 6), (4, 9), (5, 7), (6, 5), (8, 3)]

Centers = [(4, 5), (7, 4)]

print("Q7:", Distortion(Centers, Data))

# Q8 Give the center of gravity of the following (three-dimensional) data points. Enter your answer in the form (x, y, z). (Please note the space between the coordinates.)

Data = [(1, 3, -1), (9, 8, 14), (6, 2, 10), (4, 3, 1)]

n = len(Data)
centroid = tuple(sum(p[dim] for p in Data) / n for dim in range(3))

print("Q8:", centroid)

Q6: 5.0
Q7: 6.000000000000001
Q8: (5.0, 4.0, 6.0)
